[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rslab-ntua/MSc_GBDA/blob/master/2025/GBDA_2025_Lab5.ipynb)

In [ ]:
!wget https://pithos.okeanos.grnet.gr/public/FiBAB54cGZuQfXrQL7ylK -O ucf101_top5.tar.gz
!mkdir lab5_data
!tar xf ucf101_top5.tar.gz --directory lab5_data

In [ ]:
# Cell 1: Import necessary libraries
import torch
from torch.utils.data import Dataset, DataLoader
from typing import Callable, List
import pandas as pd
import os
import numpy as np
import pickle
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from tqdm.notebook import tqdm

# Define data root directory
DATA_ROOT = "lab5_data/"

# Define a Video-transform type
VTransform = Callable[[torch.Tensor], torch.Tensor]

In [ ]:
# Cell 2: Custom Dataset Implementation
class UCF101(Dataset):
    """
    UCF101 Dataset class for video classification

    This dataset handler:
    - Loads video data from a specified directory
    - Supports train/test modes
    - Applies custom video transformations
    - Implements feature precomputation for efficiency
    - Handles video frame processing and label mapping

    Key Features:
    - Lazy loading of videos to manage memory
    - Caching support for preprocessed features
    - Flexible transform pipeline
    - Automatic category indexing
    """

    def __init__(
        self,
        data_root: str,
        mode: str = "train",
        video_transforms: List[VTransform] = [],
        use_precomputed: bool = True,
    ):
        """
        Initialize the UCF101 dataset

        Args:
            data_root (str): Root directory containing the dataset
            mode (str): Either 'train' or 'test'
            video_transforms (List[VTransform]): List of transforms to apply to videos
            use_precomputed (bool): Whether to cache computed features
        """
        super().__init__()
        assert mode in ["train", "test"], "Mode must be either 'train' or 'test'"

        self.root = data_root
        self.mode = mode
        self.v_transforms = video_transforms

        # Setup precomputation caching
        self.pre = use_precomputed
        self.pre_root = os.path.join(self.root, "precomp")
        if self.pre and not os.path.exists(self.pre_root):
            os.makedirs(self.pre_root)

        # Initialize the database
        self._build_db()

    def _build_db(self):
        """
        Builds the dataset index from CSV files
        - Reads the CSV containing video paths and labels
        - Creates a mapping of category names to numerical indices
        - Stores the database as numpy array for efficient access
        """
        csv_file = os.path.join(self.root, self.mode + ".csv")
        self.db: np.ndarray = pd.read_csv(csv_file, header=0).values

        # Create category-to-index mapping
        unique_categories = np.sort(np.unique(self.db.T[1]))
        self.categories = {
            c_name: c_idx for c_idx, c_name in enumerate(unique_categories)
        }

    def compute_sample(self, video_name: str, category: str):
        """
        Process a single video sample

        Args:
            video_name (str): Name of the video file
            category (str): Category label of the video

        Returns:
            tuple: (processed_video_tensor, category_index)
        """
        # Load video frames using OpenCV for more control
        cap = cv2.VideoCapture(os.path.join(self.root, self.mode, video_name))
        frames = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            # Convert BGR to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()

        # Convert to tensor and adjust dimensions
        V = torch.tensor(np.array(frames))
        V = V.permute(0, 3, 1, 2)  # NxCxHxW

        # Apply transforms
        for T in self.v_transforms:
            V = T(V)

        return V, self.categories[category]

    def __getitem__(self, index: int):
        """
        Retrieve a specific sample from the dataset

        Args:
            index (int): Index of the sample to retrieve

        Returns:
            tuple: (processed_video_tensor, category_index)
        """
        video_name, category = self.db[index]
        cache_key = "_".join([self.mode, video_name])
        cache_path = os.path.join(self.pre_root, f"{cache_key}.tmp")

        # Try loading from cache first
        if os.path.exists(cache_path):
            with open(cache_path, "rb") as f:
                sample = pickle.load(f)
        else:
            sample = self.compute_sample(video_name, category)
            # Cache the processed sample
            if self.pre:
                with open(cache_path, "wb") as f:
                    pickle.dump(sample, f)

        return sample

    def __len__(self) -> int:
        """Returns the total number of samples in the dataset"""
        return self.db.shape[0]

In [ ]:
# Cell 3: Feature Extraction Implementation
class FeatureExtractor(nn.Module):
    """
    Custom feature extractor using ResNet architecture

    This module:
    - Uses a vanilla PyTorch ResNet implementation
    - Extracts features from the average pooling layer
    - Provides a clean interface for feature extraction
    """

    def __init__(self):
        super().__init__()
        # Create ResNet18 backbone
        self.backbone = self._create_backbone()
        self.eval()  # Set to evaluation mode

    def _create_backbone(self):
        """Creates and modifies ResNet backbone"""
        # Initialize ResNet18
        model = torch.hub.load("pytorch/vision:v0.10.0", "resnet18", pretrained=True)
        # Remove the final classification layer
        return nn.Sequential(*list(model.children())[:-1])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Extract features from input tensor

        Args:
            x (torch.Tensor): Input tensor of shape (N, C, H, W)

        Returns:
            torch.Tensor: Features of shape (N, 512)
        """
        with torch.no_grad():
            features = self.backbone(x)
            return torch.flatten(features, 1)


def compute_features() -> VTransform:
    """
    Creates a video transform function for feature extraction

    Returns:
        Callable: Transform function that extracts features from video frames
    """
    extractor = FeatureExtractor()

    def apply(v: torch.Tensor) -> torch.Tensor:
        return extractor(v)

    return apply

In [ ]:
# Cell 4: Data Loading Setup & Initial Data Visualization (before feature extraction)
def pad_sequences_collate_fn(samples: List[tuple]) -> tuple:
    """
    Custom collate function for padding sequences in a batch

    Args:
        samples (List[tuple]): List of (data, label) pairs

    Returns:
        tuple: (padded_data, labels, padding_mask)
    """
    labels = torch.stack([torch.tensor(v[1]) for v in samples])
    data = nn.utils.rnn.pad_sequence([v[0] for v in samples], batch_first=True)

    # Create padding mask
    key_mask = nn.utils.rnn.pad_sequence(
        [torch.zeros(v[0].shape[0], dtype=torch.bool) for v in samples],
        padding_value=True,
        batch_first=True,
    )

    return data, labels, key_mask


def visualize_first_frames(dataset, num_samples=4, title="Dataset Samples"):
    """
    Visualize first frames from videos in the dataset
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.ravel()

    # Get random indices
    indices = np.random.choice(len(dataset), num_samples, replace=False)

    for idx, ax in zip(indices, axes):
        # Get video name and category
        video_name, category = dataset.db[idx]

        # Load video using OpenCV
        cap = cv2.VideoCapture(os.path.join(dataset.root, dataset.mode, video_name))
        ret, frame = cap.read()
        cap.release()

        if ret:
            # Convert BGR to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Display frame
            ax.imshow(frame)
            ax.set_title(f"Category: {category}")
            ax.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


# Create datasets without transforms first
train_dset_raw = UCF101(DATA_ROOT, "train", video_transforms=[])
val_dset_raw = UCF101(DATA_ROOT, "test", video_transforms=[])

# Visualize samples from both datasets
print("Training Dataset Samples:")
visualize_first_frames(train_dset_raw, title="Training Dataset Samples")

print("\nValidation Dataset Samples:")
visualize_first_frames(val_dset_raw, title="Validation Dataset Samples")

# Now proceed with feature extraction and transform application
train_transforms = [
    lambda x: x.float() / 255.0,  # Convert to float and normalize
    lambda x: F.interpolate(x, size=(224, 224)),  # Resize
    lambda x: (x - torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
    / torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1),  # Normalize
    compute_features(),
]

# Define train/val datasets with transforms
train_dset = UCF101(DATA_ROOT, "train", video_transforms=train_transforms)
val_dset = UCF101(DATA_ROOT, "test", video_transforms=train_transforms)

# Create dataloaders
train_dloader = DataLoader(
    train_dset,
    batch_size=32,
    shuffle=True,
    collate_fn=pad_sequences_collate_fn,
    num_workers=4,
)

val_dloader = DataLoader(
    val_dset,
    batch_size=32,
    shuffle=False,
    collate_fn=pad_sequences_collate_fn,
    num_workers=4,
)

In [ ]:
# Cell 5: Positional Encoding Implementation
class PositionalEncoding(nn.Module):
    """
    Implements positional encoding for transformer architecture

    This module:
    - Adds positional information to input embeddings
    - Uses sine and cosine functions of different frequencies
    - Helps model understand sequential order of inputs

    Key Features:
    - Fixed positional encodings (non-learnable)
    - Supports variable sequence lengths up to max_len
    - Includes dropout for regularization
    """

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        """
        Initialize positional encoding

        Args:
            d_model (int): Dimension of the model embeddings
            dropout (float): Dropout rate
            max_len (int): Maximum sequence length to support
        """
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create positional encoding matrix
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        # Initialize positional encoding buffer
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)  # Even dimensions
        pe[0, :, 1::2] = torch.cos(position * div_term)  # Odd dimensions

        # Register as buffer (won't be updated during backprop)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Add positional encoding to input tensor

        Args:
            x: Input tensor of shape [batch_size, seq_len, embedding_dim]

        Returns:
            Tensor with positional encoding added
        """
        # Expand positional encoding to batch size
        pe = self.pe[:, : x.size(1)].expand(x.shape[0], -1, -1)

        # Add positional encoding and apply dropout
        return self.dropout(x + pe)

In [ ]:
# Cell 6: Transformer Model Implementation
class VideoTransformer(nn.Module):
    """
    Transformer-based model for video classification

    Architecture:
    1. Feature embedding layer
    2. Positional encoding
    3. CLS token for sequence aggregation
    4. Transformer encoder
    5. Classification head

    Key Features:
    - Uses CLS token approach similar to BERT
    - Handles variable-length sequences
    - Includes attention mechanism for temporal modeling
    """

    def __init__(self, num_classes: int, d_model: int = 128):
        """
        Initialize the transformer model

        Args:
            num_classes (int): Number of output classes
            d_model (int): Dimension of the model embeddings
        """
        super().__init__()

        # Project input features to model dimension
        self.embedding_layer = nn.Linear(512, d_model)

        # Learnable CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # Positional encoding layer
        self.pos_encoding = PositionalEncoding(d_model)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, batch_first=True, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer, num_layers=3
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, num_classes),
        )

    def forward(self, x: torch.Tensor, key_mask: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model

        Args:
            x: Input tensor of shape [batch_size, seq_len, feature_dim]
            key_mask: Padding mask for sequences

        Returns:
            Classification logits
        """
        # Project features to model dimension
        x = self.embedding_layer(x)  # [B, T, d_model]

        # Add positional encoding
        x = self.pos_encoding(x)

        # Expand CLS token to batch size and concatenate
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        # Update attention mask for CLS token
        key_mask = torch.cat(
            [
                torch.zeros(x.shape[0], 1, dtype=torch.bool, device=key_mask.device),
                key_mask,
            ],
            dim=1,
        )

        # Apply transformer encoder
        x = self.transformer(x, src_key_padding_mask=key_mask)

        # Use CLS token output for classification
        x = x[:, 0]  # Take CLS token representation

        return self.classifier(x)

In [ ]:
# Cell 7: Training Setup and Utilities
class VideoClassificationTrainer:
    """
    Trainer class for video classification model

    Features:
    - Training and validation loops
    - Metrics tracking
    - Visualization utilities
    - Model checkpointing
    """

    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        learning_rate=1e-3,
        num_epochs=50,
        device="cuda",
        checkpoint_dir="./checkpoints",
    ):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.num_epochs = num_epochs

        self.optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        self.criterion = nn.CrossEntropyLoss()

        # Metrics tracking
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []

        # Checkpointing
        self.checkpoint_dir = checkpoint_dir
        if not os.path.exists(checkpoint_dir):
            os.makedirs(checkpoint_dir)

        self.best_val_acc = 0
        self.best_val_loss = float("inf")

    def train_epoch(self):
        """Run one epoch of training"""
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0

        # Create progress bar
        pbar = tqdm(
            self.train_loader,
            desc="Training",
            position=0,
            leave=False,  # don't leave the progress bar
            dynamic_ncols=True,
        )  # automatically adjust to terminal width

        for batch_idx, (data, labels, key_mask) in enumerate(pbar):
            data, labels = data.to(self.device), labels.to(self.device)
            key_mask = key_mask.to(self.device)

            self.optimizer.zero_grad()
            outputs = self.model(data, key_mask)
            loss = self.criterion(outputs, labels)

            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            # Update progress bar
            current_loss = total_loss / (batch_idx + 1)
            current_acc = 100.0 * correct / total
            pbar.set_postfix(
                {"loss": f"{current_loss:.4f}", "acc": f"{current_acc:.2f}%"}
            )

        return total_loss / len(self.train_loader), 100.0 * correct / total

    def validate(self):
        """Run validation"""
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0

        # Create progress bar
        pbar = tqdm(
            self.val_loader,
            desc="Validating",
            position=1,
            leave=False,
            dynamic_ncols=True,
        )

        with torch.no_grad():
            for batch_idx, (data, labels, key_mask) in enumerate(pbar):
                data, labels = data.to(self.device), labels.to(self.device)
                key_mask = key_mask.to(self.device)

                outputs = self.model(data, key_mask)
                loss = self.criterion(outputs, labels)

                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

                # Update progress bar
                current_loss = total_loss / (batch_idx + 1)
                current_acc = 100.0 * correct / total
                pbar.set_postfix(
                    {"loss": f"{current_loss:.4f}", "acc": f"{current_acc:.2f}%"}
                )

        return total_loss / len(self.val_loader), 100.0 * correct / total

    def train(self):
        """Main training loop"""
        # Create progress bar for epochs
        epoch_pbar = tqdm(
            range(self.num_epochs),
            desc="Training Progress",
            position=1,
            dynamic_ncols=True,
        )

        for epoch in epoch_pbar:
            train_loss, train_acc = self.train_epoch()
            val_loss, val_acc = self.validate()

            # Store metrics
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.train_accuracies.append(train_acc)
            self.val_accuracies.append(val_acc)

            # Check if this is the best model
            is_best = False
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                is_best = True
            elif val_acc == self.best_val_acc and val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                is_best = True

            # Save checkpoint
            self.save_checkpoint(epoch + 1, val_loss, val_acc, is_best)

            # Update epoch progress bar
            epoch_pbar.set_postfix(
                {
                    "train_loss": f"{train_loss:.4f}",
                    "train_acc": f"{train_acc:.2f}%",
                    "val_loss": f"{val_loss:.4f}",
                    "val_acc": f"{val_acc:.2f}%",
                    "best": "*" if is_best else "",
                }
            )

            if is_best:
                print("New best model saved!")

    def save_checkpoint(self, epoch, val_loss, val_acc, is_best=False):
        """
        Save model checkpoint

        Args:
            epoch (int): Current epoch number
            val_loss (float): Validation loss
            val_acc (float): Validation accuracy
            is_best (bool): Whether this is the best model so far
        """
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
            "train_losses": self.train_losses,
            "val_losses": self.val_losses,
            "train_accuracies": self.train_accuracies,
            "val_accuracies": self.val_accuracies,
        }

        # Save regular checkpoint
        checkpoint_path = os.path.join(
            self.checkpoint_dir, f"checkpoint_epoch_{epoch}.pt"
        )
        torch.save(checkpoint, checkpoint_path)

        # Save best model separately
        if is_best:
            best_path = os.path.join(self.checkpoint_dir, "best_model.pt")
            torch.save(checkpoint, best_path)

    def load_checkpoint(self, checkpoint_path):
        """
        Load model checkpoint

        Args:
            checkpoint_path (str): Path to checkpoint file
        """
        checkpoint = torch.load(checkpoint_path)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        # Restore training history
        self.train_losses = checkpoint["train_losses"]
        self.val_losses = checkpoint["val_losses"]
        self.train_accuracies = checkpoint["train_accuracies"]
        self.val_accuracies = checkpoint["val_accuracies"]

        return checkpoint["epoch"]

    def plot_metrics(self):
        """Plot training metrics"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

        # Plot losses
        ax1.plot(self.train_losses, label="Train Loss")
        ax1.plot(self.val_losses, label="Val Loss")
        ax1.set_title("Training and Validation Loss")
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss")
        ax1.legend()

        # Plot accuracies
        ax2.plot(self.train_accuracies, label="Train Accuracy")
        ax2.plot(self.val_accuracies, label="Val Accuracy")
        ax2.set_title("Training and Validation Accuracy")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Accuracy (%)")
        ax2.legend()

        plt.tight_layout()
        plt.show()

In [ ]:
# Cell 8: Training Execution and Visualization
# Initialize model and trainer
model = VideoTransformer(num_classes=len(train_dset.categories))
trainer = VideoClassificationTrainer(
    model=model,
    train_loader=train_dloader,
    val_loader=val_dloader,
    learning_rate=1e-3,
    num_epochs=5,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

# Train the model
trainer.train()

# Plot training metrics
trainer.plot_metrics()


# Visualization of model predictions
def visualize_predictions(model, val_loader, num_samples=4):
    """
    Visualize model predictions on a batch from validation loader

    Args:
        model: Trained model
        val_loader: Validation dataloader
        num_samples: Number of samples to visualize (max 4)
    """
    model.eval()

    # Get a single batch
    data, labels, key_mask = next(iter(val_loader))

    # Move to device
    data = data.to("cuda" if torch.cuda.is_available() else "cpu")
    key_mask = key_mask.to("cuda" if torch.cuda.is_available() else "cpu")

    # Get predictions
    with torch.no_grad():
        preds = model(data, key_mask)
        pred_classes = preds.argmax(1)

    # Get category mapping
    categories = {v: k for k, v in val_loader.dataset.categories.items()}

    # Create figure
    num_vis = min(num_samples, 32)
    fig, axes = plt.subplots(num_vis, 2, figsize=(12, 4 * num_vis))

    # Handle single sample case
    if num_vis == 1:
        axes = axes.reshape(1, -1)

    for i in range(num_vis):
        # Get features for this sample
        features = data[i]  # Shape: [T, feature_dim]

        # Create a placeholder image for visualization
        # Since we're working with features, we'll show a representation
        plt.sca(axes[i, 0])
        plt.imshow(features[:, :100].cpu().numpy(), aspect="auto", cmap="viridis")
        plt.title("Feature Visualization\n(first 100 dimensions)")
        plt.colorbar()

        axes[i, 0].set_ylim(0, 100)

        # Plot prediction vs ground truth
        pred_text = f"Predicted: {categories[pred_classes[i].item()]}\n"
        true_text = f"Actual: {categories[labels[i].item()]}"
        color = "green" if pred_classes[i].item() == labels[i].item() else "red"

        axes[i, 1].text(
            0.5,
            0.5,
            pred_text + true_text,
            ha="center",
            va="center",
            color=color,
            fontsize=12,
            bbox=dict(facecolor="white", alpha=0.8),
        )
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()


# Visualize some predictions
visualize_predictions(model, val_dloader)